# 01 — Data Ingestion and Optimization

This notebook loads the six raw Instacart CSV files into Databricks, applies explicit schemas, handles missing product metadata, and saves optimized Parquet tables for downstream analysis and modeling.

In [0]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType
)

base_path = "/Volumes/workspace/default/instacart"  # adjust if needed

aisles_schema = StructType([
    StructField("aisle_id", IntegerType(), True),
    StructField("aisle", StringType(), True)
])

departments_schema = StructType([
    StructField("department_id", IntegerType(), True),
    StructField("department", StringType(), True)
])

products_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("aisle_id", IntegerType(), True),
    StructField("department_id", IntegerType(), True)
])

orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("eval_set", StringType(), True),
    StructField("order_number", IntegerType(), True),
    StructField("order_dow", IntegerType(), True),
    StructField("order_hour_of_day", IntegerType(), True),
    StructField("days_since_prior_order", DoubleType(), True)
])

order_products_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("add_to_cart_order", IntegerType(), True),
    StructField("reordered", IntegerType(), True)
])

aisles = spark.read.csv(f"{base_path}/aisles.csv", header=True, schema=aisles_schema)
departments = spark.read.csv(f"{base_path}/departments.csv", header=True, schema=departments_schema)
products = spark.read.csv(f"{base_path}/products.csv", header=True, schema=products_schema)
orders = spark.read.csv(f"{base_path}/orders.csv", header=True, schema=orders_schema)
prior = spark.read.csv(f"{base_path}/order_products__prior.csv", header=True, schema=order_products_schema)
train = spark.read.csv(f"{base_path}/order_products__train.csv", header=True, schema=order_products_schema)

## Why explicit schemas?

The raw CSVs are large enough that relying on `inferSchema=True` can be slow and error-prone. Explicit schemas make the ingestion pipeline faster, reproducible, and safer for production-style processing.

In [0]:
from pyspark.sql import functions as F

# Fix product metadata before joining
products_fixed = (
    products
    .withColumn("aisle_id", F.coalesce(F.col("aisle_id"), F.lit(-1)))
    .withColumn("department_id", F.coalesce(F.col("department_id"), F.lit(-1)))
)

aisles_fixed = aisles.unionByName(
    spark.createDataFrame([(-1, "unknown")], ["aisle_id", "aisle"])
)

departments_fixed = departments.unionByName(
    spark.createDataFrame([(-1, "unknown")], ["department_id", "department"])
)

product_details_fixed = (
    products_fixed
    .join(aisles_fixed, on="aisle_id", how="left")
    .join(departments_fixed, on="department_id", how="left")
)

prior_order_details_fixed = (
    prior
    .join(orders, on="order_id", how="left")
    .join(product_details_fixed, on="product_id", how="left")
)

train_order_details_fixed = (
    train
    .join(orders, on="order_id", how="left")
    .join(product_details_fixed, on="product_id", how="left")
)

In [0]:
tables = {
    "aisles": aisles,
    "departments": departments,
    "products": products,
    "orders": orders,
    "prior": prior,
    "train": train
}

for name, df in tables.items():
    print(name, df.count())

In [0]:
for name, df in tables.items():
    print(f"\n{name.upper()}")
    df.printSchema()

In [0]:
optimized_path = f"{base_path}/optimized"

dbutils.fs.mkdirs(optimized_path)

In [0]:
aisles_fixed.write.mode("overwrite").parquet(f"{optimized_path}/aisles")
departments_fixed.write.mode("overwrite").parquet(f"{optimized_path}/departments")
products_fixed.write.mode("overwrite").parquet(f"{optimized_path}/products")
orders.write.mode("overwrite").parquet(f"{optimized_path}/orders")
prior.write.mode("overwrite").parquet(f"{optimized_path}/order_products_prior")
train.write.mode("overwrite").parquet(f"{optimized_path}/order_products_train")

In [0]:
aisles_pq = spark.read.parquet(f"{optimized_path}/aisles")
departments_pq = spark.read.parquet(f"{optimized_path}/departments")
products_pq = spark.read.parquet(f"{optimized_path}/products")
orders_pq = spark.read.parquet(f"{optimized_path}/orders")
prior_pq = spark.read.parquet(f"{optimized_path}/order_products_prior")
train_pq = spark.read.parquet(f"{optimized_path}/order_products_train")

product_details = (
    products_pq
    .join(aisles_pq, on="aisle_id", how="left")
    .join(departments_pq, on="department_id", how="left")
)

prior_order_details = (
    prior_pq
    .join(orders_pq, on="order_id", how="left")
    .join(product_details, on="product_id", how="left")
)

train_order_details = (
    train_pq
    .join(orders_pq, on="order_id", how="left")
    .join(product_details, on="product_id", how="left")
)

In [0]:
silver_path = f"{base_path}/silver"

dbutils.fs.mkdirs(silver_path)

product_details.write.mode("overwrite").parquet(f"{silver_path}/product_details")
prior_order_details.write.mode("overwrite").parquet(f"{silver_path}/prior_order_details")
train_order_details.write.mode("overwrite").parquet(f"{silver_path}/train_order_details")

## Missing metadata handling

One product had missing `aisle_id` and `department_id` in the raw product table. Rather than manually assigning a category, the missing values were replaced with sentinel ID `-1` and labeled as `"unknown"`. This preserves data integrity and keeps the pipeline scalable and reproducible.

In [0]:
missing_check = (
    spark.read.parquet(f"{silver_path}/prior_order_details")
    .select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in ["product_id", "product_name", "aisle_id", "department_id", "aisle", "department"]
    ])
)

display(missing_check)

In [0]:
print("product_details:", spark.read.parquet(f"{silver_path}/product_details").count())
print("prior_order_details:", spark.read.parquet(f"{silver_path}/prior_order_details").count())
print("train_order_details:", spark.read.parquet(f"{silver_path}/train_order_details").count())

In [0]:
display(
    spark.read.parquet(f"{silver_path}/prior_order_details")
    .select(
        "user_id",
        "order_id",
        "order_number",
        "product_id",
        "product_name",
        "aisle",
        "department",
        "add_to_cart_order",
        "reordered",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order"
    )
    .limit(20)
)